In [3]:
# adiciona a raiz do repo no sys.path (funciona no VS Code e no Colab)
from pathlib import Path
import sys


repo_root = Path().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
if (repo_root / "src").exists():
    sys.path.insert(0, str(repo_root))
else:
    raise RuntimeError("Não encontrei a pasta 'src' acima do diretório atual.")


In [4]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from src.models.mlp_activations import select_mlp_activations
from src.pixel_preprocessing import (
    prepare_pixel_data,
    standardize_bands,
    apply_pca
)
from keras.models import Sequential
from keras.layers import Dense, Input, Flatten
import tensorflow as tf
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


&emsp;Os hiperparâmetros do modelo foram definidos priorizando estabilidade, capacidade de generalização e baixo custo computacional. A taxa de aprendizado foi fixada em 0,001, valor padrão recomendado para o otimizador Adam, que proporciona convergência estável e evita oscilações grandes durante o treinamento; valores maiores poderiam gerar instabilidade, enquanto valores menores tornariam o processo mais lento. O otimizador escolhido foi o Adam (Adaptive Moment Estimation), devido à sua capacidade de combinar as vantagens do Momentum e do RMSProp, ajustando automaticamente a taxa de aprendizado para cada parâmetro, resultando em convergência rápida e robusta, sendo amplamente utilizado como referência em redes neurais. O tamanho do lote (batch size) foi definido como 32, equilibrando estabilidade na atualização dos gradientes e custo computacional, além de reduzir o ruído nas atualizações em comparação a batches muito pequenos. O modelo foi treinado por 100 épocas, número suficiente para permitir que a rede convergisse sem overfitting inicial, podendo ser ajustado posteriormente com técnicas como Early Stopping se necessário. Veja na tabela a seguir os hiperparâmetros definidos:


  <strong>Tabela  1</strong> – Hiperparâmetros definidos


| Hiperparâmetro                       | Valor Definido                                  | Justificativa                                           |
| ------------------------------------ | ----------------------------------------------- | ------------------------------------------------------- |
| Taxa de aprendizado (learning rate)  | 0.001                                           | Valor padrão estável para otimizadores adaptativos      |
| Otimizador                           | Adam                                            | Convergência rápida e robusta                           |
| Batch size                           | 32                                              | Bom equilíbrio entre estabilidade e custo computacional |
| Número de épocas                     | 100                                             | Suficiente para convergência sem overfitting inicial    |
| Função de ativação (camadas ocultas) | ReLU                                            | Evita problema de gradiente desaparecendo               |
| Função de ativação (saída)           | Sigmoid                          | Adequado para classificação binária                          |
| Função de perda                      | Binary Crossentropy  | Adequado para classificação binária                     |
| Número de camadas ocultas            |  2                                          | Arquitetura simples para baseline                       |
| Número de neurônios                  | 64                                      | Capacidade suficiente sem excesso de complexidade       |
<p style="font-size: 0.9em;">Fonte: Elaboração própria.</p>



&emsp;A arquitetura escolhida é simples, com duas camadas ocultas contendo 64 neurônios na primeira camada e 32 na segunda , garantindo capacidade de modelar relações não-lineares enquanto mantém baixo risco de overfitting e custo computacional reduzido. Para as camadas ocultas, a função de ativação adotada foi a ReLU, que evita o problema de gradientes desaparecendo e acelera o treinamento, sendo padrão em redes densas. Para a camada de saída, utilizou-se Sigmoid como função de saída, por ser mais adequada para probelmas de classificação binária. A função de perda escolhida foi Binary Crossentropy, também considerando o tipo de classificação.


In [5]:
def load_dataset_with_groups(dataset_path: Path, codes_path: Path):

    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset nao encontrado: {dataset_path}")

    if not codes_path.exists():
        raise FileNotFoundError(f"Arquivo de codigos nao encontrado: {codes_path}")

    df = pd.read_csv(dataset_path)

    with open(codes_path, "r", encoding="utf-8") as f:
        codes = json.load(f)

    positivos = set(codes.get("positivos", []))
    negativos = set(codes.get("negativos", []))

    all_ids = sorted(list(positivos | negativos), key=len, reverse=True)
    pattern = "|".join(re.escape(v) for v in all_ids)

    df["image_id"] = df["path"].astype(str).str.extract(rf"({pattern})", expand=False)

    def map_label(img_id):
        if img_id in positivos:
            return 1
        if img_id in negativos:
            return 0
        return np.nan

    df["label"] = df["image_id"].map(map_label)
    df = df.dropna(subset=["image_id", "label"]).copy()
    df["label"] = df["label"].astype(int)

    feature_cols = [c for c in df.columns if c.startswith("pixel_")]

    return df, feature_cols

In [6]:

DATASET_PATH = repo_root / "data" / "pixels_dataset.csv"
CODES_PATH = repo_root / "data" / "extracted_codes.json"
OUTPUT_DIR = repo_root / "outputs" / "a03_mlp_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repositorio: {repo_root}")
print(f"Dataset: {DATASET_PATH}")
print(f"Codigos: {CODES_PATH}")
print(f"Saida: {OUTPUT_DIR}")

Repositorio: C:\Users\Inteli\Documents\g01
Dataset: C:\Users\Inteli\Documents\g01\data\pixels_dataset.csv
Codigos: C:\Users\Inteli\Documents\g01\data\extracted_codes.json
Saida: C:\Users\Inteli\Documents\g01\outputs\a03_mlp_baseline


In [7]:
df_all, feature_cols = load_dataset_with_groups(
    DATASET_PATH,
    CODES_PATH
)

print("Dataset carregado:", df_all.shape)
print("Número de features:", len(feature_cols))

Dataset carregado: (295, 147466)
Número de features: 147456


In [8]:
n_pixels = len(feature_cols)
n_bands = 9

pixels_per_band = n_pixels // n_bands
side = int(np.sqrt(pixels_per_band))

print("Total colunas pixel:", n_pixels)
print("Pixels por banda:", pixels_per_band)
print("Dimensão estimada:", side, "x", side)

Total colunas pixel: 147456
Pixels por banda: 16384
Dimensão estimada: 128 x 128


In [9]:
SEED = 42

image_level = df_all[["image_id", "label"]].drop_duplicates()

train_val_ids, test_ids = train_test_split(
    image_level["image_id"],
    test_size=0.20,
    random_state=SEED,
    stratify=image_level["label"],
)

train_val_df = image_level[image_level["image_id"].isin(train_val_ids)]

train_ids, val_ids = train_test_split(
    train_val_df["image_id"],
    test_size=0.25,
    random_state=SEED,
    stratify=train_val_df["label"],
)

df_train = df_all[df_all["image_id"].isin(train_ids)].copy()
df_val   = df_all[df_all["image_id"].isin(val_ids)].copy()
df_test  = df_all[df_all["image_id"].isin(test_ids)].copy()

In [10]:
band_cols = feature_cols  # suas 147456 colunas

def compute_band_means(df, n_bands=9):
    n_pixels = len(band_cols)
    pixels_per_band = n_pixels // n_bands
    
    X = df[band_cols].values
    X = X.reshape(len(df), n_bands, pixels_per_band)
    
    band_means = X.mean(axis=2)  # média por banda
    
    columns = [f"B{i+1:02d}" for i in range(n_bands)]
    df_means = pd.DataFrame(band_means, columns=columns)
    df_means["label"] = df["label"].values
    
    return df_means

df_train_spec = compute_band_means(df_train)
df_val_spec   = compute_band_means(df_val)
df_test_spec  = compute_band_means(df_test)

In [11]:
scaler = StandardScaler()

X_train = scaler.fit_transform(df_train_spec.drop(columns="label"))
X_val   = scaler.transform(df_val_spec.drop(columns="label"))
X_test  = scaler.transform(df_test_spec.drop(columns="label"))

y_train = df_train_spec["label"].values
y_val   = df_val_spec["label"].values
y_test  = df_test_spec["label"].values

In [12]:
pca = PCA(n_components=0.95, random_state=SEED)

X_train_pca = pca.fit_transform(X_train)
X_val_pca   = pca.transform(X_val)
X_test_pca  = pca.transform(X_test)

print("Componentes finais:", pca.n_components_)

Componentes finais: 2


In [13]:
INPUT_SHAPE = (X_train_pca.shape[1],)

In [14]:
TASK_TYPE = "classification"
N_CLASSES = 2   # alterar para >2 caso multiclasse

In [15]:
activation_config = select_mlp_activations(
    task_type=TASK_TYPE,
    n_classes=N_CLASSES
)

hidden_activation = activation_config["hidden_activation"]
output_activation = activation_config["output_activation"]

print("Configuração selecionada:")
print(activation_config)

Configuração selecionada:
{'hidden_activation': 'relu', 'hidden_justification': 'ReLU nas camadas ocultas acelera o treinamento e reduz saturacao de gradiente em comparacao com Sigmoid.', 'output_activation': 'sigmoid', 'output_justification': 'Sigmoid na saida e adequada para classificacao binaria, produzindo probabilidade no intervalo [0, 1].'}


In [16]:
from tensorflow.keras import regularizers

def build_mlp(input_shape, n_classes, hidden_activation, output_activation):

    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Dense(
            64,
            activation=hidden_activation,
            kernel_regularizer=regularizers.l2(0.001)
        ),
        layers.Dropout(0.3),

        layers.Dense(
            32,
            activation=hidden_activation,
            kernel_regularizer=regularizers.l2(0.001)
        ),
        layers.Dropout(0.3),

        layers.Dense(
            1 if n_classes == 2 else n_classes,
            activation=output_activation
        )
    ])

    return model

In [17]:
model = build_mlp(
    input_shape=INPUT_SHAPE,
    n_classes=N_CLASSES,
    hidden_activation=hidden_activation,
    output_activation=output_activation
)

In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,305 (9.00 KB)

 Trainable params: 2,305 (9.00 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

if N_CLASSES == 2:
    loss_fn = "binary_crossentropy"
    metrics = [
        "accuracy",
        tf.keras.metrics.AUC(name="auc_roc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
else:
    loss_fn = "sparse_categorical_crossentropy"
    metrics = [
        "accuracy",
        tf.keras.metrics.AUC(name="auc_roc", multi_label=True)
    ]

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=metrics
)

In [20]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [21]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("Model input shape:", model.input_shape)
print("Número de classes:", len(np.unique(y_train)))

X_train shape: (177, 9)
y_train shape: (177,)
Model input shape: (None, 2)
Número de classes: 2


In [23]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("Model input shape:", model.input_shape)
print("Número de classes:", len(np.unique(y_train)))

X_train shape: (177, 9)
y_train shape: (177,)
Model input shape: (None, 2)
Número de classes: 2


In [24]:
from tensorflow.keras.backend import clear_session
clear_session()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy as np

INPUT_SHAPE = (X_train.shape[1],)

model = Sequential([
    Dense(32, activation='relu', input_shape=INPUT_SHAPE),
    Dense(16, activation='relu'),
    Dense(2, activation='sigmoid')  # 2 classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

c:\Users\Inteli\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [26]:
print("Model input shape:", model.input_shape)

Model input shape: (None, 9)


In [27]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.6045 - loss: 0.6332 - val_accuracy: 0.6102 - val_loss: 0.6218
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6045 - loss: 0.6057 - val_accuracy: 0.6102 - val_loss: 0.6061
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6045 - loss: 0.5851 - val_accuracy: 0.6102 - val_loss: 0.5972
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6045 - loss: 0.5691 - val_accuracy: 0.6441 - val_loss: 0.5930
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7175 - loss: 0.5570 - val_accuracy: 0.7119 - val_loss: 0.5901
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8023 - loss: 0.5473 - val_accuracy: 0.7627 - val_loss: 0.5868
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8023 - loss: 0.5386 - val_accuracy: 0.7627 - val_loss: 0.5849
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7966 - loss: 0.5311 - val_accuracy: 0.7627 - val_loss: